# 01 Statistics

Feature pruning (428 attributes to 74) and the statistics behind Extended Data Fig. 2. Attributes and summaries per network come from `data/features_per_network`; if a network is missing there, `build_predictors_from_original` below rebuilds it from the map files, which needs the map deposit (`data/RAW_SOURCES.md`). Results go to `StatResults`.

Choices to note:
- each of the three same-named city pairs (London, Birmingham, Manchester) is paired with its own map files, which carry each other's names;
- Graz's and Innsbruck's precipitation columns (node and network level) are set to missing, so the median fills them: their GeoQuery request returned temperature instead;
- the 74 retained features are mapped to normal scores before the multivariate models (PLS, elastic net, LASSO, random forest). On the raw scale a few heavy-tailed attributes made held-out predictions explode (out-of-fold R2 near -50,000). Spearman correlations are rank-based and unchanged.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))
from paths import FEATURES_PER_NETWORK, OUTPUTS_DIR, RAW_LINKS, RAW_MAPS, RESULTS

# --- Fast MFD FPCA-style analysis with per-city caching (TXT outputs + original CSV inputs) ---
import os, glob, warnings
warnings.filterwarnings(
    "ignore",
    message="divide by zero encountered in scalar divide",
    module="statsmodels.stats.outliers_influence"
)
from typing import Dict, List, Tuple, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import statsmodels.api as sm
from sklearn.base import clone
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.dummy import DummyRegressor

# =======================
# Collinearity & FDR utils
# =======================
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import KFold
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNetCV
from sklearn.multioutput import MultiOutputRegressor
LINKS_DIR   = RAW_LINKS   # read only for a network missing from data/features_per_network
NODES_DIR   = RAW_MAPS    # likewise
# ------------- Config knobs -------------
ALLOW_PARALLEL = False
N_JOBS = -1
USE_ONLY_CLEAN_FEATURES = True
USE_ALL_DATAPOINTS = False      # True = use all cities (current behavior). False = hold out ~20 for validation.
HOLDOUT_N = 20                 # number of validation cities when USE_ALL_DATAPOINTS is False
SPLIT_RANDOM_STATE = 42        # reproducible split

def load_outputs_for_city(outputs_dir: str, city_name: str) -> Tuple[Dict[str,float], np.ndarray]:
    path = os.path.join(outputs_dir, f"outputs_{city_name}.txt")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing outputs file for {city_name}: {path}")
    with open(path, "r", encoding="utf-8") as fh:
        txt = fh.read().strip()
    parts = [float(s) for s in txt.split(",")]
    if len(parts) < 7:
        raise ValueError(f"Expected 7 outputs in {path}, got {len(parts)}")
    # Reuse your existing extractor to build dict + flow vector
    y = np.array(parts[:7], dtype=float)
    responses, qrel = extract_responses(y)
    return responses, qrel
def variance_threshold_filter(df: pd.DataFrame, thr: float = 1e-8) -> pd.DataFrame:
    # Remove (near-)constant predictors
    if df.shape[1] == 0:
        return df
    vt = VarianceThreshold(threshold=thr)
    X = vt.fit_transform(df.values)
    kept = df.columns[vt.get_support(indices=True)]
    return df.loc[:, kept]

def drop_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Remove exactly duplicated columns
    if df.shape[1] == 0:
        return df
    dup_mask = df.T.duplicated()
    return df.loc[:, ~dup_mask]

def correlation_prune(df: pd.DataFrame, thresh: float = 0.95) -> Tuple[pd.DataFrame, List[Tuple[str,str,float]]]:
    """
    Greedy correlation pruning on predictors using |rho| > thresh.
    For a correlated pair, drop the one with higher average absolute correlation to others.
    Returns pruned df and list of dropped pairs with rho.
    """
    if df.shape[1] <= 1:
        return df, []
    corr = df.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = set()
    dropped_pairs = []
    while True:
        # Find the strongest remaining correlation
        max_val = upper.max().max()
        if not np.isfinite(max_val) or max_val <= thresh:
            break
        i, j = np.where(upper.values == max_val)
        i, j = int(i[0]), int(j[0])
        col_i, col_j = upper.index[i], upper.columns[j]
        # drop the one more correlated on average
        mean_i = corr[col_i].mean()
        mean_j = corr[col_j].mean()
        drop_col = col_i if mean_i >= mean_j else col_j
        keep_col = col_j if drop_col == col_i else col_i
        to_drop.add(drop_col)
        dropped_pairs.append((keep_col, drop_col, float(max_val)))
        # update masks
        upper.loc[:, drop_col] = np.nan
        upper.loc[drop_col, :] = np.nan
    kept = [c for c in df.columns if c not in to_drop]
    return df.loc[:, kept], dropped_pairs

def compute_vif(df: pd.DataFrame) -> pd.Series:
    """
    Fast VIF via inverse correlation matrix:
      VIF_j = (R^{-1})_{jj} for standardized predictors.
    IMPORTANT: If X is rank-deficient (e.g., p >= n or perfect multicollinearity),
    OLS-based VIFs are undefined/infinite. We detect that and return inf to trigger pruning.
    """
    p = df.shape[1]
    if p == 0:
        return pd.Series(dtype=float)
    if p == 1:
        return pd.Series([1.0], index=df.columns, dtype=float)

    X = df.values.astype(float)
    n = X.shape[0]

    # --- NEW: guard for high-dimensional / singular design ---
    # Center to assess rank properly
    Xc = X - np.nanmean(X, axis=0, keepdims=True)
    # If p >= n-1 or matrix rank < p, treat VIFs as infinite to force pruning
    # (matches statsmodels behavior in singular designs)
    try:
        rnk = np.linalg.matrix_rank(Xc, tol=1e-10)
    except Exception:
        rnk = p  # fallback: assume full rank
    if p >= max(n - 1, 1) or rnk < p:
        return pd.Series(np.inf, index=df.columns, dtype=float)

    # Standardize (guard against zeros)
    mean = np.nanmean(X, axis=0)
    std = np.nanstd(X, axis=0, ddof=0)
    std_safe = std.copy()
    std_safe[std_safe == 0] = np.nan  # mark constant cols

    Xz = (X - mean) / std_safe
    const_mask = ~np.isfinite(std_safe)  # columns with zero variance

    # Build correlation matrix
    Xz = np.nan_to_num(Xz, nan=0.0, posinf=0.0, neginf=0.0)
    R = np.corrcoef(Xz, rowvar=False)

    # Fill any diagonal NaNs (can occur for constant columns)
    diag = np.diag(R)
    bad_diag = ~np.isfinite(diag)
    if bad_diag.any():
        for i in np.where(bad_diag)[0]:
            R[i, i] = 1.0

    # If R is nearly singular, pinv is fine—but it yields finite numbers.
    # We already guarded with the rank check above so we’re in the full-rank regime here.
    Rinv = np.linalg.inv(R) if np.isfinite(R).all() else np.linalg.pinv(R, rcond=1e-8)

    vifs = np.diag(Rinv).astype(float)
    vifs[const_mask] = np.inf           # constant columns => VIF = inf
    vifs = np.maximum(vifs, 1.0)        # clip tiny negatives from numerical noise
    return pd.Series(vifs, index=df.columns, dtype=float)


def vif_prune(df: pd.DataFrame, max_vif: float = 10.0) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """
    Iteratively remove a single column at a time until all VIFs <= max_vif.
    If any VIFs are non-finite (inf/NaN), remove exactly one of those columns,
    chosen as the one with the highest mean absolute correlation with the rest.
    Returns (X_pruned, vifs_final, dropped_cols)
    """
    X = df.copy()
    dropped: List[str] = []

    def pick_most_redundant(colnames: List[str], Xsub: pd.DataFrame) -> str:
        """Among 'colnames', return the one with highest mean |corr| with other kept columns."""
        if len(Xsub.columns) <= 1:
            return colnames[0]
        # safe corr (fill NaNs with 0 to avoid degenerate cases)
        C = Xsub.corr().abs().fillna(0.0)
        means = {}
        for c in colnames:
            others = [k for k in Xsub.columns if k != c]
            if len(others) == 0:
                means[c] = 0.0
            else:
                means[c] = float(C.loc[c, others].mean())
        # tiebreaker: deterministic by (mean desc, name asc)
        return sorted(means.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

    while X.shape[1] > 1:
        vifs = compute_vif(X)  # may contain inf/NaN when perfect collinearity exists

        # Case A: some VIFs are non-finite -> drop exactly ONE of them (most redundant)
        bad_mask = ~np.isfinite(vifs.values)
        if bad_mask.any():
            bad_cols = vifs.index[bad_mask].tolist()
            victim = pick_most_redundant(bad_cols, X)
            X = X.drop(columns=[victim])
            dropped.append(victim)
            continue

        # Case B: all finite -> drop the single worst if above threshold
        vmax = float(vifs.max())
        if vmax <= max_vif:
            # Done
            return X, vifs, dropped

        worst = str(vifs.idxmax())
        X = X.drop(columns=[worst])
        dropped.append(worst)

    # If we got down to 1 column, VIF is by definition 1.0 (see compute_vif)
    return X, compute_vif(X), dropped

def benjamini_hochberg(pvals: pd.Series, alpha: float = 0.05) -> pd.Series:
    """
    Return boolean series: True if p-value is significant under BH-FDR at level alpha.
    """
    p = pvals.values.astype(float)
    n = p.size
    order = np.argsort(p)
    ranked = np.empty_like(order)
    ranked[order] = np.arange(1, n+1)
    crit = (ranked / n) * alpha
    sig = p <= crit
    # enforce monotonicity from largest p downwards
    sig_sorted = sig[order]
    for i in range(n-2, -1, -1):
        sig_sorted[i] = sig_sorted[i] or sig_sorted[i+1]
    out = np.zeros_like(sig, dtype=bool)
    out[order] = sig_sorted
    return pd.Series(out, index=pvals.index)

def attach_fdr(df_corr: pd.DataFrame) -> pd.DataFrame:
    """
    Given a correlation table with 'p_value', add BH-FDR decisions and q-values (Storey q via BH approx).
    """
    if df_corr.empty:
        return df_corr
    df = df_corr.copy()
    df["is_significant_bh_0.05"] = benjamini_hochberg(df["p_value"], alpha=0.05)
    # rough BH q-value: q = p * n / rank
    n = df.shape[0]
    order = np.argsort(df["p_value"].values)
    ranks = np.empty_like(order)
    ranks[order] = np.arange(1, n+1)
    qvals = df["p_value"].values * n / ranks
    # make non-decreasing
    q_sorted = qvals[order]
    for i in range(n-2, -1, -1):
        q_sorted[i] = min(q_sorted[i], q_sorted[i+1])
    q = np.empty_like(qvals)
    q[order] = q_sorted
    df["q_bh"] = q
    return df

# =======================
# Collinearity cleaner with back-off
# =======================
def collinearity_cleanup_with_backoff(
    X_in: pd.DataFrame,
    corr_thresholds=(0.95, 0.90, 0.85),         # try conservative -> stricter
    max_vif_thresholds=(10.0, 8.0, 6.0, 5.0),   # try looser -> stricter
    min_features=10,                             # try to keep at least this many
    var_thr=1e-12
):
    """
    Run variance filter, duplicate drop, correlation prune, VIF prune with
    progressively stricter thresholds only as needed.

    Returns:
      X_clean,
      logs: dict with counts and parameters used,
      dropped_pairs (from corr step),
      vifs_final (Series),
      removed_by_step: dict(step_name -> List[str] of removed columns)
    """
    logs = {}
    removed_by_step: Dict[str, List[str]] = {}

    # -- Start --
    X0 = X_in.select_dtypes(include=[np.number]).copy()
    logs["n_start"] = int(X0.shape[1])

    # 1) near-zero variance
    X1 = variance_threshold_filter(X0, thr=var_thr)
    var_removed = pd.Index(X0.columns).difference(X1.columns).tolist()
    removed_by_step["variance"] = var_removed
    logs["n_after_variance"] = int(X1.shape[1])
    logs["dropped_variance"] = int(len(var_removed))

    # 2) drop exact duplicate columns
    X2 = drop_duplicate_columns(X1)
    dedup_removed = pd.Index(X1.columns).difference(X2.columns).tolist()
    removed_by_step["deduplicate"] = dedup_removed
    logs["n_after_dedup"] = int(X2.shape[1])
    logs["dropped_dedup"] = int(len(dedup_removed))

    # 3) correlation prune with back-off
    dropped_pairs_all = []
    X3_best, pairs_best, thr_used = X2, [], None
    for t in corr_thresholds:
        X3_try, pairs_try = correlation_prune(X2, thresh=t)
        # accept this level if it preserves enough features, or it's the last try
        if X3_try.shape[1] >= min_features or t == corr_thresholds[-1]:
            X3_best, pairs_best, thr_used = X3_try, pairs_try, t
            break
    logs["corr_threshold_used"] = float(thr_used)
    logs["n_after_corr"] = int(X3_best.shape[1])

    # derive corr-removed explicitly
    corr_removed_from_pairs = {drop for (_, drop, _) in pairs_best}
    corr_removed_residual = set(X2.columns) - set(X3_best.columns)
    corr_removed = sorted(set(corr_removed_from_pairs) | set(corr_removed_residual))
    removed_by_step["correlation"] = list(corr_removed)
    logs["dropped_correlation"] = int(len(corr_removed))
    dropped_pairs_all.extend(pairs_best)

    # 4) VIF prune with back-off
    X4_best, vifs_best, vif_used = X3_best, None, None
    vif_removed_this_level: List[str] = []
    for mv in max_vif_thresholds:
        X4_try, vifs_try, dropped_vif = vif_prune(X3_best, max_vif=mv)
        # accept this level if it preserves enough features, or it's the last try
        if X4_try.shape[1] >= min_features or mv == max_vif_thresholds[-1]:
            X4_best, vifs_best, vif_used = X4_try, vifs_try, mv  # <-- use the threshold value
            vif_removed_this_level = dropped_vif
            break
    logs["max_vif_used"] = float(vif_used)
    logs["n_after_vif"] = int(X4_best.shape[1])
    logs["dropped_vif"] = int(len(vif_removed_this_level))
    removed_by_step["vif"] = list(vif_removed_this_level)

    # Failsafe: if we somehow reached 0, fall back to post-dedup
    if X4_best.shape[1] == 0:
        X4_best = X2.copy()
        vifs_best = compute_vif(X4_best)
        logs["fallback_to"] = "post_dedup"
        logs["n_after_fallback"] = int(X4_best.shape[1])

    # Summary roll-up
    logs["n_final"] = int(X4_best.shape[1])
    logs["total_dropped"] = int(sum(len(v) for v in removed_by_step.values()))

    return X4_best, logs, dropped_pairs_all, vifs_best, removed_by_step

# =======================
# Multivariate modeling (PLS2 + VIP) and ElasticNet (multioutput)
# =======================
def select_pls_components_cv(X: np.ndarray, Y: np.ndarray, max_comp: int = 10, n_splits: int = 5, random_state: int = 42) -> int:
    max_comp = max(1, min(max_comp, X.shape[1], Y.shape[1]))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    best_k, best_err = 1, np.inf
    for k in range(1, max_comp+1):
        errs = []
        for tr, te in kf.split(X):
            pls = PLSRegression(n_components=k, scale=True)
            pls.fit(X[tr], Y[tr])
            Yp = pls.predict(X[te])
            err = np.nanmean((Yp - Y[te])**2)  # MSE across all outputs
            errs.append(err)
        m = float(np.nanmean(errs))
        if m < best_err:
            best_err, best_k = m, k
    return best_k

def pls_vip(pls: PLSRegression, X: np.ndarray, Y: np.ndarray) -> pd.Series:
    """
    Variable Importance in Projection (VIP) for multi-response (PLS2).

    VIP_j = sqrt( p * sum_k( SSY_k * (w_{j,k}^2 / ||w_k||^2) ) / sum_k SSY_k )
    where:
      - p         = number of predictors
      - k         = component index
      - SSY_k     = sum_i t_{i,k}^2 * sum_m q_{m,k}^2  (per-component explained SS of Y)
      - w_{j,k}   = x-weight for predictor j on component k
      - ||w_k||^2 = sum_j w_{j,k}^2
    """
    T = pls.x_scores_          # (n_samples, h)
    W = pls.x_weights_         # (p, h)
    Q = pls.y_loadings_        # (m, h)

    p, h = W.shape
    if h == 0:
        return pd.Series(np.zeros((0,)))

    # Per-component SSY (length h)
    sumsq_t = np.sum(T**2, axis=0)      # (h,)
    sumsq_q = np.sum(Q**2, axis=0)      # (h,)
    ssy = sumsq_t * sumsq_q             # (h,)
    total_ssy = float(np.sum(ssy))

    # Guard against degenerate cases
    if not np.isfinite(total_ssy) or total_ssy <= 0:
        return pd.Series(np.nan, index=np.arange(p))

    # ||w_k||^2 per component (length h), avoid divide-by-zero
    wnorm2 = np.sum(W**2, axis=0)       # (h,)
    wnorm2 = np.where(wnorm2 == 0, np.finfo(float).eps, wnorm2)

    vip = np.zeros(p, dtype=float)
    for j in range(p):
        # sum over components: SSY_k * (w_{j,k}^2 / ||w_k||^2)
        contrib = ssy * (W[j, :]**2 / wnorm2)
        vip[j] = np.sqrt(p * np.sum(contrib) / total_ssy)

    return pd.Series(vip)


def fit_pls2_report(X: np.ndarray, Y: np.ndarray, feature_names: List[str], out_dir: str, tag: str = "multi",
                    X_eval: np.ndarray = None, Y_eval: np.ndarray = None):
    # Pick #components by CV
    k = select_pls_components_cv(X, Y, max_comp=10)
    pls = PLSRegression(n_components=k, scale=True)
    pls.fit(X, Y)
    Y_hat_train = pls.predict(X)

    # R² names
    y_names = [f"Y{i}" for i in range(Y.shape[1])]

    # Train R² (always saved to a separate *train* file)
    r2_train = []
    for i in range(len(y_names)):
        num = ((Y[:, i] - Y_hat_train[:, i])**2).sum()
        den = ((Y[:, i] - Y[:, i].mean())**2).sum()
        r2_train.append(1.0 - (num/den if den > 0 else np.nan))
    pd.DataFrame({"response": y_names, "r2_in_sample": r2_train}).to_csv(
        os.path.join(out_dir, f"pls2_{tag}_r2_train.csv"), index=False
    )

    # Eval R² (validation if provided; otherwise same as train/in-sample)
    if X_eval is not None and Y_eval is not None and X_eval.shape[0] > 0:
        Y_hat_eval = pls.predict(X_eval)
        r2_eval = []
        for i in range(len(y_names)):
            num = ((Y_eval[:, i] - Y_hat_eval[:, i])**2).sum()
            den = ((Y_eval[:, i] - Y_eval[:, i].mean())**2).sum()
            r2_eval.append(1.0 - (num/den if den > 0 else np.nan))
        pd.DataFrame({"response": y_names, "r2_in_sample": r2_eval}).to_csv(
            os.path.join(out_dir, f"pls2_{tag}_r2.csv"), index=False
        )
    else:
        # Keep current behavior (all-data in-sample)
        pd.DataFrame({"response": y_names, "r2_in_sample": r2_train}).to_csv(
            os.path.join(out_dir, f"pls2_{tag}_r2.csv"), index=False
        )

    # Coefficients
    coef = pls.coef_
    expected = (len(feature_names), len(y_names))
    if coef.shape != expected:
        # Some sklearn builds expose coef_ as (n_targets, n_features)
        coef = coef.T
    coef_df = pd.DataFrame(coef, index=feature_names, columns=y_names)
    coef_df.to_csv(os.path.join(out_dir, f"pls2_{tag}_coefficients.csv"))

    # VIP
    vip = pls_vip(pls, X, Y)
    vip.index = feature_names
    vip.name = "VIP"
    vip.sort_values(ascending=False).to_csv(os.path.join(out_dir, f"pls2_{tag}_vip.csv"))
    # Components chosen
    with open(os.path.join(out_dir, f"pls2_{tag}_summary.txt"), "w", encoding="utf-8") as fh:
        fh.write(f"Selected PLS components (CV): {k}\n")
        fh.write("Higher VIP (~>1) often indicates influential predictors on the multivariate response.\n")

def fit_elasticnet_multi(X: np.ndarray, Y: np.ndarray, feature_names: List[str], out_dir: str, tag: str = "multi"):
    # Elastic Net with scaling inside a pipeline
    from sklearn.pipeline import make_pipeline
    base = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,1.0],
                     alphas=100, cv=5, random_state=42, max_iter=100000)
    )
    enet = MultiOutputRegressor(base)
    enet.fit(X, Y)

    # Collect standardized coefs, original-scale coefs, intercepts, and hyperparams
    std_coefs = []
    orig_coefs = []
    orig_intercepts = []
    chosen_alpha = []
    chosen_l1ratio = []

    for est in enet.estimators_:
        scaler = est.named_steps["standardscaler"]
        model  = est.named_steps["elasticnetcv"]

        beta_std = model.coef_.astype(float)         # standardized-X coefficients
        b0_std   = float(model.intercept_)           # intercept on standardized-X scale

        # Map back to original X units:
        # If X_std = (X - mean)/scale, then  y = b0_std + beta_std @ X_std
        # => y = (b0_std - sum_j beta_std_j * mean_j/scale_j) + sum_j (beta_std_j/scale_j) * X_j
        scale = scaler.scale_.astype(float)
        mean  = scaler.mean_.astype(float)

        # Guard against division by zero for any zero-variance features (shouldn't happen after your cleanup)
        scale_safe = np.where(scale == 0.0, 1.0, scale)

        beta_orig = beta_std / scale_safe
        b0_orig   = b0_std - np.dot(beta_std, mean / scale_safe)

        std_coefs.append(beta_std)
        orig_coefs.append(beta_orig)
        orig_intercepts.append(b0_orig)
        chosen_alpha.append(model.alpha_)
        # l1_ratio can be array-like if CV searched; pick the winning one
        chosen_l1ratio.append(float(np.atleast_1d(model.l1_ratio_)[0]))

    # Shape to [n_features, n_targets]
    std_mat  = np.column_stack(std_coefs)
    orig_mat = np.column_stack(orig_coefs)

    y_names = [f"Y{i}" for i in range(Y.shape[1])]

    # Save standardized-X coefficients (comparable across features)
    pd.DataFrame(std_mat, index=feature_names, columns=y_names).to_csv(
        os.path.join(out_dir, f"elasticnet_{tag}_coefficients_standardizedX.csv")
    )

    # Save original-unit coefficients + intercepts (nice for interpretation)
    pd.DataFrame(orig_mat, index=feature_names, columns=y_names).to_csv(
        os.path.join(out_dir, f"elasticnet_{tag}_coefficients_original_units.csv")
    )
    pd.DataFrame({"response": y_names,
                  "intercept_original_units": orig_intercepts,
                  "alpha": chosen_alpha,
                  "l1_ratio": chosen_l1ratio}).to_csv(
        os.path.join(out_dir, f"elasticnet_{tag}_intercepts_and_hyperparams.csv"),
        index=False
    )

# ------------------------
# Targets
# ------------------------
def extract_responses(y: np.ndarray) -> Tuple[Dict[str, float], np.ndarray]:
    """
    y = [ qc=q(kc), kc, q(kc/6), q(kc/3), q(kc/2), q(2kc/3), q(5kc/6) ]
    Returns:
      - responses dict with ALL 7 values (for responses.csv)
      - flow vector at normalized densities: [q(kc/6), q(kc/3), q(kc/2), q(2kc/3), q(5kc/6), q(kc)]
    """
    if y.ndim != 1 or y.size < 7:
        raise ValueError("y must be a 1D vector with at least 7 elements.")
    qc = float(y[0]); kc = float(y[1])
    q_kc_over_6  = float(y[2])
    q_kc_over_3  = float(y[3])
    q_kc_over_2  = float(y[4])
    q_2kc_over_3 = float(y[5])
    q_5kc_over_6 = float(y[6])

    responses = {
        "qc": qc,
        "kc": kc,
        "q_kc/6":  q_kc_over_6,
        "q_kc/3":  q_kc_over_3,
        "q_kc/2":  q_kc_over_2,
        "q_2kc/3": q_2kc_over_3,
        "q_5kc/6": q_5kc_over_6
    }
    qrel = np.array([q_kc_over_6, q_kc_over_3, q_kc_over_2, q_2kc_over_3, q_5kc_over_6, qc], dtype=float)
    return responses, qrel

# the map files of three city pairs carry each other's names. "London" holds the map around the WebTRIS sites, whose
# curve is "London2"; "London2" holds the map around UTD19's detectors, whose curve is "london". Windows ignores case.
MAP_NAME = {"london": "London2", "London2": "London", "birmingham": "Birmingham2", "Birmingham2": "Birmingham",
            "manchester": "Manchester2", "Manchester2": "Manchester"}


def build_predictors_from_original(city_name: str, links_dir: str, nodes_dir: str) -> Dict[str, float]:
    """
    Build predictors directly from original CSVs:
      - Links: aggregate per-link categorical, binary, numeric features.
      - Nodes: aggregate per-node numeric features.
      - Network_Properties: read city-level numerical features.
    """

    # --- 1. Links ---
    link_path = os.path.join(links_dir, f"links_{MAP_NAME.get(city_name, city_name)}.csv")
    links = pd.read_csv(link_path)

    feats = {}

    # Handle categoricals: one-hot encode Class and Access, then aggregate (fractions)
    for catcol in ["Class", "Access"]:
        dummies = pd.get_dummies(links[catcol], prefix=f"link_{catcol}", dummy_na=False)
        feats.update((dummies.mean()).to_dict())   # fraction of links in each category

    for bcol in ["Bridge","Tunnel"]:
        if links[bcol].dtype == object:
            vals = links[bcol].astype(str).str.strip().str.lower().map({"yes":1, "no":0})
        else:
            vals = links[bcol]
        feats[f"link_{bcol}_frac"] = vals.fillna(0).mean()


    for bcol in ["SpeedLimit_missing","Oneway","Roundabout"]:
        feats[f"link_{bcol}_frac"] = links[bcol].mean()

    # Handle numeric (int/float): summarize with mean, std, min, max, median
    for numcol in ["Lanes","num_detectors","LinkLength","Curvature","SpeedLimit","avg_detector_pos"]:
        feats[f"link_{numcol}_mean"] = links[numcol].replace(-1, np.nan).mean()
        feats[f"link_{numcol}_std"]  = links[numcol].replace(-1, np.nan).std()
        feats[f"link_{numcol}_min"]  = links[numcol].replace(-1, np.nan).min()
        feats[f"link_{numcol}_max"]  = links[numcol].replace(-1, np.nan).max()
        feats[f"link_{numcol}_q50"] = links[numcol].replace(-1, np.nan).median()

    # --- 2. Nodes ---
    node_path = os.path.join(nodes_dir, f"nodes_{MAP_NAME.get(city_name, city_name)}.csv")
    nodes = pd.read_csv(node_path)
    # GeoQuery returned monthly temperature, not precipitation, for Austria;
    # treat Graz's and Innsbruck's *_precip values as missing (median-filled below).
    if city_name.lower() in ("graz", "innsbruck"):
        nodes[[c for c in nodes.columns if c.endswith("_precip")]] = np.nan

    # skip NodeID, Longitude, Latitude → summarize 63 numeric variables
    for col in nodes.columns:
        if col in ["NodeID","Longitude","Latitude"]:
            continue
        feats[f"node_{col}_mean"] = nodes[col].mean()
        feats[f"node_{col}_std"]  = nodes[col].std()
        feats[f"node_{col}_min"]  = nodes[col].min()
        feats[f"node_{col}_max"]  = nodes[col].max()

    # --- 3. Network-level props ---
    net_props = pd.read_csv(os.path.join(nodes_dir, "Network_Properties.csv"))

    def norm(s): 
        return str(s).strip().lower().replace(" ", "_")

    m = net_props["City"].apply(norm) == norm(MAP_NAME.get(city_name, city_name))
    if not m.any():
        raise ValueError(f"Network_Properties: no row matching city '{city_name}'")

    # keep the original numeric columns
    row = net_props.loc[m].drop(columns=["City"]).squeeze()
    if city_name.lower() in ("graz", "innsbruck"):  # same as the node level
        row[[c for c in row.index if str(c).endswith("_precip")]] = np.nan

    # --- NEW FEATURE: Density (ppl/km^2) ---
    # Expect these exact column names in Network_Properties.csv:
    #   "City Population" and "Area (km^2)"
    pop = pd.to_numeric(row.get("City Population"), errors="coerce")
    area = pd.to_numeric(row.get("Area (km^2)"), errors="coerce")

    if pd.notna(pop) and pd.notna(area) and float(area) > 0:
        feats["Density (ppl/km^2)"] = float(pop) / float(area)
    else:
        # if area is zero/missing, set NaN so it gets median-imputed later
        feats["Density (ppl/km^2)"] = np.nan

    # bring all the original network properties in as well
    feats.update(row.to_dict())

    feats["city"] = city_name
    return feats

def cache_paths(cache_dir: str, name: str):
    return (
        os.path.join(cache_dir, f"{name}__predictors.csv"),
        os.path.join(cache_dir, f"{name}__responses.csv"),
        os.path.join(cache_dir, f"{name}__flow.npy"),
    )

def is_cached(cache_dir: str, name: str) -> bool:
    p1, p2, p3 = cache_paths(cache_dir, name)
    return os.path.exists(p1) and os.path.exists(p2) and os.path.exists(p3)

def save_cache(cache_dir: str, name: str, predictors: Dict[str, float], responses: Dict[str, float], flow: np.ndarray):
    os.makedirs(cache_dir, exist_ok=True)
    pd.DataFrame([predictors]).to_csv(cache_dir + f"/{name}__predictors.csv", index=False)
    pd.DataFrame([responses]).to_csv(cache_dir + f"/{name}__responses.csv", index=False)
    np.save(cache_dir + f"/{name}__flow.npy", flow)

def load_cache(cache_dir: str, name: str):
    preds = pd.read_csv(cache_dir + f"/{name}__predictors.csv")
    resps = pd.read_csv(cache_dir + f"/{name}__responses.csv")
    flow = np.load(cache_dir + f"/{name}__flow.npy")
    return preds, resps, flow

def process_one_city(city_name: str, cache_dir: str, force: bool = False):
    name = city_name  # keep naming consistent with cache
    if (not force) and is_cached(cache_dir, name):
        return "cached", name, None
    try:
        predictors = build_predictors_from_original(
            name,
            links_dir=LINKS_DIR,
            nodes_dir=NODES_DIR
        )
        responses, qrel = load_outputs_for_city(OUTPUTS_DIR, name)
        save_cache(cache_dir, name, predictors, responses, qrel)
        return "ok", name, None
    except Exception as e:
        warnings.warn(f"Failed to process {name}: {e}")
        return "failed", name, str(e)

# ------------------------
# Modeling / analysis
# ------------------------
def run_pca_on_flows(flow_matrix: np.ndarray, n_components: int = 3, normalize_by_qc: bool = False, qc_vec: np.ndarray = None):
    X = flow_matrix.copy()
    if normalize_by_qc:
        if qc_vec is None:
            qc_vec = flow_matrix[:, -1]
        qc_safe = np.where(qc_vec == 0, 1.0, qc_vec)
        X = X / qc_safe[:, None]
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = scaler.fit_transform(X)
    pca = PCA(n_components=n_components, random_state=42)
    scores = pca.fit_transform(Xs)
    return {
        "scaler": scaler,
        "pca": pca,
        "scores": scores,
        "loadings": pca.components_,
        "explained": pca.explained_variance_ratio_,
        "X_scaled": Xs,
        "X_used": X
    }

def spearman_corr_table(X_df: pd.DataFrame, y_series: pd.Series, min_non_nan: int = 30) -> pd.DataFrame:
    rows = []
    y = y_series.values
    for col in X_df.columns:
        x = X_df[col].values
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() >= min_non_nan:
            rho, p = stats.spearmanr(x[mask], y[mask])
            rows.append({"predictor": col, "rho": rho, "p_value": p, "n": int(mask.sum())})
    out = pd.DataFrame(rows).sort_values(by="rho", key=lambda s: s.abs(), ascending=False)
    return out

def lasso_then_ols(X: np.ndarray, y: np.ndarray, feature_names: List[str], alphas: List[float] = None, cv: int = 5) -> Dict[str, Any]:
    if alphas is None:
        alphas = np.logspace(-3, 1, 30)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = scaler.fit_transform(X)
    lasso = LassoCV(alphas=alphas, cv=cv, random_state=42, n_jobs=None, max_iter=100000)
    lasso.fit(Xs, y)
    coef = lasso.coef_
    selected_idx = np.where(np.abs(coef) > 1e-8)[0]
    selected_features = [feature_names[i] for i in selected_idx]
    if len(selected_idx) > 0:
        X_sel = X[:, selected_idx]
        X_ols = sm.add_constant(X_sel, has_constant='add')
        ols_model = sm.OLS(y, X_ols, missing="drop").fit()
        ols_summary = ols_model.summary().as_text()
    else:
        ols_summary = "No features selected by LASSO (all coefficients shrank to zero)."
    return {
        "selected_features": selected_features,
        "lasso_alpha": lasso.alpha_,
        "lasso_coef": dict(zip(feature_names, coef)),
        "ols_summary": ols_summary
    }

def random_forest_importance(X: np.ndarray, y: np.ndarray, feature_names: List[str], n_estimators: int = 300, random_state: int = 42) -> pd.DataFrame:
    rf = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state, n_jobs=-1)
    rf.fit(X, y)
    perm = permutation_importance(rf, X, y, n_repeats=20, random_state=random_state, n_jobs=-1)
    imp_df = pd.DataFrame({
        "predictor": feature_names,
        "perm_importance_mean": perm.importances_mean,
        "perm_importance_std": perm.importances_std
    }).sort_values(by="perm_importance_mean", ascending=False)
    return imp_df

def _safe_name(s: str) -> str:
    return s.replace("/", "_").replace(" ", "_")

def cv_r2_multi(model, X, Y, n_splits=5, random_state=42):
    """
    Generic out-of-fold R² per response for any sklearn multi-output model.
    Returns (r2_per_response, Y_pred_oof).
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    Y_pred = np.zeros_like(Y, dtype=float)

    for tr, te in kf.split(X):
        mdl = clone(model)
        mdl.fit(X[tr], Y[tr])
        Y_pred[te] = mdl.predict(X[te])

    r2s = [r2_score(Y[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    return np.array(r2s, dtype=float), Y_pred

def nested_cv_r2_pls(X, Y, max_comp=10, n_splits=5, random_state=42):
    """
    Outer CV to get OOF predictions. Inside each training fold,
    pick PLS #components using your existing select_pls_components_cv on the *training* data only.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    Y_pred = np.zeros_like(Y, dtype=float)

    for tr, te in kf.split(X):
        # select k using only training fold
        k_best = select_pls_components_cv(X[tr], Y[tr], max_comp=max_comp, n_splits=n_splits, random_state=random_state)
        pls = PLSRegression(n_components=k_best, scale=True)
        pls.fit(X[tr], Y[tr])
        Y_pred[te] = pls.predict(X[te])

    r2s = [r2_score(Y[:, i], Y_pred[:, i]) for i in range(Y.shape[1])]
    return np.array(r2s, dtype=float), Y_pred

# ------------------------
# End-to-end runner
# ------------------------
def run_pipeline(out_dir: str,
                 n_pca: int = 3,
                 normalize_by_qc: bool = True,
                 force_recompute: bool = False):
    os.makedirs(out_dir, exist_ok=True)
    cache_dir = FEATURES_PER_NETWORK
    os.makedirs(cache_dir, exist_ok=True)

    # Discover cities from the outputs directory: outputs_{city}.txt
    out_files = sorted(glob.glob(os.path.join(OUTPUTS_DIR, "outputs_*.txt")))
    if not out_files:
        raise FileNotFoundError(f"No outputs_*.txt files found in {OUTPUTS_DIR}")

    cities = [os.path.splitext(os.path.basename(p))[0][len("outputs_"):] for p in out_files]
    print(f"Found {len(cities)} cities from outputs. Caching summaries to: {cache_dir}")

    # Process per-city with caching
    statuses = []
    if ALLOW_PARALLEL:
        from joblib import Parallel, delayed
        res = Parallel(n_jobs=N_JOBS, backend="loky", verbose=10)(
            delayed(process_one_city)(city, cache_dir, force=force_recompute) for city in cities
        )
        statuses.extend(res)
    else:
        for city in tqdm(cities):
            statuses.append(process_one_city(city, cache_dir, force=force_recompute))

    # Collect cached outputs
    preds_rows, resp_rows, flows = [], [], []
    for name in cities:
        if not is_cached(cache_dir, name):
            continue
        preds, resps, flow = load_cache(cache_dir, name)
        preds["city"] = name
        resps["city"] = name
        preds_rows.append(preds)
        resp_rows.append(resps)
        flows.append(flow.reshape(1, -1))

    if not preds_rows:
        raise RuntimeError("No cached summaries found (all cities failed?)")

    # Concatenate tables
    predictors_df = pd.concat(preds_rows, ignore_index=True).set_index("city")
    if predictors_df.shape[0] != 141:
        print(f"[WARN] Expected 141 cities, got {predictors_df.shape[0]}. See missing_present_cities.csv")
        pd.Series(predictors_df.index, name="present_cities").to_csv(os.path.join(out_dir, "present_cities.csv"))
    responses_df = pd.concat(resp_rows, ignore_index=True).set_index("city")
    flow_matrix = np.vstack(flows)

    # Clean predictor columns: drop all-NaN columns
    predictors_df = predictors_df.dropna(axis=1, how="all")
    # Zero-fill one-hot fractions that are missing in a city
    cat_cols = [c for c in predictors_df.columns if c.startswith("link_Class_") or c.startswith("link_Access_")]
    predictors_df[cat_cols] = predictors_df[cat_cols].fillna(0.0)
    
    # ----- NEW: export the full per-city feature table (pre-statistical cleanup) -----
    all_features_raw = predictors_df.copy()
    all_features_raw.to_csv(os.path.join(out_dir, "predictors_all_cities_raw.csv"))  # 141 rows expected
    # -------------------------------------------------------------------------------

    # Now handle any remaining sporadic NaNs with medians
    if predictors_df.isna().any().any():
        predictors_df = predictors_df.fillna(predictors_df.median(numeric_only=True))


    # --- New: rigorous collinearity cleanup on predictors (with back-off & diagnostics) ---
    X_numeric = predictors_df.select_dtypes(include=[np.number]).copy()
    # Keep a copy of ALL numeric features (pre-cleanup) for optional use downstream
    X_all = X_numeric.copy()

    
    # Try to keep >= 10 features; adapt thresholds if needed.
    X_clean, clean_logs, dropped_pairs, vifs_final, removed_by_step = collinearity_cleanup_with_backoff(
        X_numeric,
        corr_thresholds=(0.95, 0.90, 0.85),
        max_vif_thresholds=(10.0, 8.0, 6.0, 5.0),
        min_features=10,
        var_thr=1e-12
    )
    
    # diagnostics
    diag_dir = os.path.join(out_dir, "diagnostics")
    os.makedirs(diag_dir, exist_ok=True)
    
    # 3.1 Summary counts per step
    summary_rows = [
        {"step": "start",        "n_after": clean_logs.get("n_start", np.nan),           "dropped": 0},
        {"step": "variance",     "n_after": clean_logs.get("n_after_variance", np.nan),  "dropped": clean_logs.get("dropped_variance", np.nan)},
        {"step": "deduplicate",  "n_after": clean_logs.get("n_after_dedup", np.nan),     "dropped": clean_logs.get("dropped_dedup", np.nan)},
        {"step": "correlation",  "n_after": clean_logs.get("n_after_corr", np.nan),      "dropped": clean_logs.get("dropped_correlation", np.nan),
                                  "param_used": f"thresh={clean_logs.get('corr_threshold_used', np.nan)}"},
        {"step": "vif",          "n_after": clean_logs.get("n_after_vif", np.nan),       "dropped": clean_logs.get("dropped_vif", np.nan),
                                  "param_used": f"max_vif={clean_logs.get('max_vif_used', np.nan)}"},
    ]
    if "fallback_to" in clean_logs:
        summary_rows.append({"step": "fallback", "n_after": clean_logs.get("n_after_fallback", np.nan), "dropped": np.nan, "param_used": clean_logs["fallback_to"]})
    summary_rows.append({"step": "final", "n_after": clean_logs.get("n_final", np.nan), "dropped": clean_logs.get("total_dropped", np.nan)})
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(diag_dir, "feature_drop_summary.csv"), index=False)
    # Also save the raw logs dict for easy inspection (keys/values)
    pd.DataFrame([clean_logs]).to_csv(os.path.join(diag_dir, "collinearity_cleanup_logs.csv"), index=False)
    
    # 3.2 Detailed lists of removed columns per step
    pd.Series(removed_by_step.get("variance", []), name="removed_by_variance").to_csv(
        os.path.join(diag_dir, "removed_by_variance.csv"), index=False
    )
    pd.Series(removed_by_step.get("deduplicate", []), name="removed_by_deduplicate").to_csv(
        os.path.join(diag_dir, "removed_by_deduplicate.csv"), index=False
    )
    pd.Series(removed_by_step.get("correlation", []), name="removed_by_correlation").to_csv(
        os.path.join(diag_dir, "removed_by_correlation.csv"), index=False
    )
    pd.Series(removed_by_step.get("vif", []), name="removed_by_vif").to_csv(
        os.path.join(diag_dir, "removed_by_vif.csv"), index=False
    )
    
    # 3.3 Correlation pairs that triggered drops
    if dropped_pairs:
        pd.DataFrame(dropped_pairs, columns=["kept","dropped","abs_corr"]).to_csv(
            os.path.join(diag_dir, "correlation_pruned_pairs.csv"), index=False
        )
    
    # 3.4 Final VIFs of the kept set
    if vifs_final is not None and not vifs_final.empty:
        vifs_final.to_csv(os.path.join(diag_dir, "final_vif.csv"))
    
    # 3.5 Kept vs removed (total view)
    kept_cols = pd.Index(X_clean.columns)
    removed_cols_total = pd.Index(X_numeric.columns).difference(kept_cols)
    pd.Series(kept_cols, name="kept").to_csv(os.path.join(diag_dir, "predictors_kept.csv"), index=False)
    pd.Series(removed_cols_total, name="removed").to_csv(os.path.join(diag_dir, "predictors_removed_total.csv"), index=False)
    
    # Also echo a one-line console summary
    print(
        f"[Diagnostics] start={clean_logs.get('n_start')}  "
        f"→ var {clean_logs.get('n_after_variance')}  "
        f"→ dedup {clean_logs.get('n_after_dedup')}  "
        f"→ corr({clean_logs.get('corr_threshold_used')}) {clean_logs.get('n_after_corr')}  "
        f"→ vif({clean_logs.get('max_vif_used')}) {clean_logs.get('n_after_vif')}  "
        f"→ final={clean_logs.get('n_final')}"
    )
    
    # Keep both versions; decide which to feed to modeling/metrics
    predictors_df_clean = X_clean.copy()
    predictors_df_all   = X_all.copy()

    X_for_model = predictors_df_clean if USE_ONLY_CLEAN_FEATURES else predictors_df_all

    # (Optional but very handy) Save all three for traceability
    predictors_df_clean.to_csv(os.path.join(out_dir, "predictors_aggregated_clean.csv"))
    predictors_df_all.to_csv(os.path.join(out_dir, "predictors_aggregated_allnumeric.csv"))
    X_for_model.to_csv(os.path.join(out_dir, "predictors_aggregated_used.csv"))

    # columns for flows
    flow_cols = ["q_kc/6","q_kc/3","q_kc/2","q_2kc/3","q_5kc/6","q_kc"]
    flow_df = pd.DataFrame(flow_matrix, columns=flow_cols, index=X_for_model.index)

    # Save raw aggregated
    predictors_df.to_csv(os.path.join(out_dir, "predictors_aggregated.csv"))
    responses_df.to_csv(os.path.join(out_dir, "responses.csv"))  # now includes all 7 outputs
    flow_df.to_csv(os.path.join(out_dir, "flow_profiles.csv"))

    # PCA on flows (discrete FPCA)
    print("Running PCA on flow profiles...")
    fpca = run_pca_on_flows(flow_matrix, n_components=n_pca, normalize_by_qc=normalize_by_qc, qc_vec=responses_df["qc"].values)
    scores = fpca["scores"]; loadings = fpca["loadings"]; explained = fpca["explained"]

    scores_cols = [f"flow_pca{i+1}" for i in range(scores.shape[1])]
    scores_df = pd.DataFrame(scores, columns=scores_cols, index=X_for_model.index)
    scores_df.to_csv(os.path.join(out_dir, "flow_pca_scores.csv"))

    loadings_df = pd.DataFrame(loadings, columns=flow_cols)
    loadings_df.to_csv(os.path.join(out_dir, "flow_pca_loadings.csv"), index_label="component")
    pd.DataFrame({"component": scores_cols, "explained_variance_ratio": explained}).to_csv(
        os.path.join(out_dir, "flow_pca_explained_variance.csv"), index=False
    )

    # Scree + loadings plots
    plt.figure()
    plt.plot(np.arange(1, len(explained)+1), explained, marker='o')
    plt.xlabel("PCA Component"); plt.ylabel("Explained Variance Ratio")
    plt.title("Flow Profile PCA Scree"); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "flow_pca_scree.png"), dpi=180); plt.close()

    dens_grid = np.array([1/6, 1/3, 1/2, 2/3, 5/6, 1.0])
    for i in range(loadings.shape[0]):
        plt.figure()
        plt.plot(dens_grid, loadings[i, :], marker='o')
        plt.xlabel("Normalized density (k/kc)"); plt.ylabel(f"Loading (PC{i+1})")
        plt.title(f"Flow PCA Component {i+1} Loadings"); plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"flow_pca_component_{i+1}_loadings.png"), dpi=180)
        plt.close()

    # --- New: build multivariate response matrix Y (all 7 outputs + flow PCs) ---
    # Y responses include original 7 outputs AND the flow PCA scores (shape variation)
    y_cols_base = ["qc","kc","q_kc/6","q_kc/3","q_kc/2","q_2kc/3","q_5kc/6"]
    Y_df = responses_df[y_cols_base].join(scores_df, how="left")
    # Ensure aligned with the chosen feature set
    Y_df = Y_df.loc[X_for_model.index]

    # Save the modeling matrices actually used (cleaned X, multiresponse Y)
    X_df = X_for_model.select_dtypes(include=[np.number]).copy()
    X_df.to_csv(os.path.join(out_dir, "modeling_X_clean.csv"))
    # several attributes are heavy-tailed (the land-cover spreads of the Seattle and
    # Portland networks lie hundreds of interquartile ranges from the median), so on the raw
    # scale the linear models extrapolate wildly for held-out networks (out-of-fold R2 near
    # -50,000). Map every feature to normal scores before the multivariate models. This uses
    # the attributes only, never the outcomes; rank-based results (Spearman, random-forest
    # splits) are unchanged by it.
    from sklearn.preprocessing import QuantileTransformer
    _qt = QuantileTransformer(n_quantiles=X_df.shape[0], output_distribution="normal")
    X_df = pd.DataFrame(_qt.fit_transform(X_df), index=X_df.index, columns=X_df.columns)
    X_df.to_csv(os.path.join(out_dir, "modeling_X_normal_scores.csv"))
    Y_df.to_csv(os.path.join(out_dir, "modeling_Y_multi.csv"))

    # Fit PLS2 with CV-selected components; export R^2, coefficients, VIP
    X = X_df.values.astype(float)
    Y = Y_df.values.astype(float)
    # Keep full copies for CV baselines (unchanged)
    X_full, Y_full = X.copy(), Y.copy()

    # Train/validation split depending on the switch
    if USE_ALL_DATAPOINTS:
        # train = all; "validation" = all (so evaluation equals in-sample)
        train_idx = np.arange(X.shape[0])
        val_idx   = np.arange(X.shape[0])
    else:
        rng = np.random.RandomState(SPLIT_RANDOM_STATE)
        n = X.shape[0]
        val_n = min(HOLDOUT_N, max(1, n // 7))  # ~20 for 141; small guard for tiny n
        all_idx = np.arange(n)
        val_idx = rng.choice(all_idx, size=val_n, replace=False)
        train_mask = np.ones(n, dtype=bool); train_mask[val_idx] = False
        train_idx = np.where(train_mask)[0]

    X_train, Y_train = X[train_idx], Y[train_idx]
    X_val,   Y_val   = X[val_idx],   Y[val_idx]

    # ---- Baseline: Dummy (predict-mean) out-of-fold R² per response
    dummy = MultiOutputRegressor(DummyRegressor(strategy="mean"))
    r2_dummy, _ = cv_r2_multi(dummy, X_full, Y_full, n_splits=5, random_state=42)
    pd.DataFrame({
        "response": [f"Y{i}" for i in range(Y.shape[1])],
        "r2_cv_dummy_mean": r2_dummy
    }).to_csv(os.path.join(out_dir, "baseline_dummy_mean_r2_CV.csv"), index=False)
    
    if X_df.shape[1] == 0:
        with open(os.path.join(out_dir, "MODEL_SKIPPED.txt"), "w", encoding="utf-8") as fh:
            fh.write(
                "All predictors were removed during collinearity cleanup.\n"
                "Check diagnostics/feature_drop_summary.csv for per-step counts and thresholds, \n"
                "diagnostics/collinearity_cleanup_logs.csv for the raw step metrics, and \n"
                "diagnostics/removed_by_*.csv for the exact columns removed at each step.\n"
                "Consider relaxing correlation (higher thresh) and/or max VIF.\n"
            )
        print("No predictors left after cleanup. Skipping PLS/ElasticNet and downstream univariate models.")
        return {
            "predictors_df": predictors_df,
            "responses_df": responses_df,
            "flow_df": flow_df,
            "scores_df": scores_df,
            "status": pd.DataFrame(statuses, columns=["status","city","error"])
        }

    # ---- Honest performance: PLS with nested CV component selection (OOF R²)
    r2_pls_cv, Y_pred_pls_oof = nested_cv_r2_pls(X_full, Y_full, max_comp=10, n_splits=5, random_state=42)
    pd.DataFrame({
        "response": [f"Y{i}" for i in range(Y.shape[1])],
        "r2_cv_oof": r2_pls_cv
    }).to_csv(os.path.join(out_dir, "pls2_all_targets_r2_CV.csv"), index=False)

    fit_pls2_report(
        X_train, Y_train,
        feature_names=list(X_df.columns),
        out_dir=out_dir, tag="all_targets",
        X_eval=(None if USE_ALL_DATAPOINTS else X_val),
        Y_eval=(None if USE_ALL_DATAPOINTS else Y_val)
    )

    # Fit Multioutput ElasticNet (robustness vs PLS)
    fit_elasticnet_multi(X_train, Y_train, feature_names=list(X_df.columns), out_dir=out_dir, tag="all_targets")
    # ---- Honest performance: ElasticNet (pipeline with imputer+scaler) out-of-fold R²
    enet_base = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ])
    # ElasticNetCV already handles alpha/l1_ratio CV; we wrap it to clone cleanly
    enet_cv = MultiOutputRegressor(
        make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(with_mean=True, with_std=True),
            ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,1.0],
                         alphas=100, cv=5, random_state=42, max_iter=100000)
        )
    )
    r2_enet_cv, Y_pred_enet_oof = cv_r2_multi(enet_cv, X_full, Y_full, n_splits=5, random_state=42)
    pd.DataFrame({
        "response": [f"Y{i}" for i in range(Y.shape[1])],
        "r2_cv_oof": r2_enet_cv
    }).to_csv(os.path.join(out_dir, "elasticnet_all_targets_r2_CV.csv"), index=False)

    print("Computing Spearman correlations...")
    # Use pre-clean numeric predictors for correlation breadth
    X_df_corr = X_for_model
    
    def save_corr(target_name: str, series: pd.Series):
        corr = spearman_corr_table(X_df_corr, series, min_non_nan=30)
        corr = attach_fdr(corr)
        fname = target_name.replace('/','_').replace(' ','_')
        corr.to_csv(os.path.join(out_dir, f"corr_predictors_vs_{fname}.csv"), index=False)


    save_corr("kc", responses_df["kc"])
    save_corr("qc", responses_df["qc"])
    for col in flow_cols:
        save_corr(col, flow_df[col])
    for col in scores_cols:
        save_corr(col, scores_df[col])

    # LASSO -> OLS & RF importances for kc + PCs
    print("Running LASSO selection + OLS and RF importances...")
    feature_names = list(X_df.columns)
    # use training rows for target vectors
    targets = {"kc": Y_train[:, y_cols_base.index("kc")]}  # kc is the 2nd column in y_cols_base
    # The flow PCA scores were appended to Y_df after the 7 base outputs. Build indices for them:
    score_start = len(y_cols_base)  # 7
    for i, c in enumerate(scores_cols):
        targets[c] = Y_train[:, score_start + i]

    # Fit on training only
    for key, y_tr in targets.items():
        res = lasso_then_ols(X_train, y_tr, feature_names=feature_names)
        with open(os.path.join(out_dir, f"lasso_ols_{key}.txt"), "w", encoding="utf-8") as fh:
            fh.write(f"# Target: {key}\n")
            fh.write(f"LASSO alpha: {res['lasso_alpha']}\n")
            fh.write("Selected features:\n")
            for fn in res["selected_features"]:
                fh.write(f"  - {fn}\n")
            fh.write("\n--- OLS Summary ---\n")
            fh.write(res["ols_summary"])

        imp_df = random_forest_importance(X_train, y_tr, feature_names=feature_names)
        imp_df.to_csv(os.path.join(out_dir, f"rf_permutation_importance_{key}.csv"), index=False)
    # === Supplemental (integrated): LASSO→OLS + RF for the 6 original flow outputs ===
    # Targets: q_kc/6, q_kc/3, q_kc/2, q_2kc/3, q_5kc/6, qc
    # Behavior:
    # - If USE_ALL_DATAPOINTS=True: fit on ALL rows and write the original filenames (backward-compatible).
    # - If USE_ALL_DATAPOINTS=False: fit on TRAIN and
    #     * write VALIDATION results to the original filenames
    #     * write TRAINING results to separate *_train.* files.

    flow_targets_order = ["q_kc/6", "q_kc/3", "q_kc/2", "q_2kc/3", "q_5kc/6", "qc"]
    feature_names = list(X_df.columns)

    def _metrics_report(y_true: np.ndarray, y_hat: np.ndarray) -> str:
        # r2_score is already imported
        r2 = float(r2_score(y_true, y_hat)) if y_true.size > 0 else np.nan
        mae = float(np.mean(np.abs(y_true - y_hat))) if y_true.size > 0 else np.nan
        rmse = float(np.sqrt(np.mean((y_true - y_hat) ** 2))) if y_true.size > 0 else np.nan
        return f"R2 = {r2:.6f}\nMAE = {mae:.6f}\nRMSE = {rmse:.6f}\n"

    for t in flow_targets_order:
        # Column index of this target inside Y_df (y_cols_base = ["qc","kc","q_kc/6","q_kc/3","q_kc/2","q_2kc/3","q_5kc/6"])
        col_idx = y_cols_base.index(t)

        if USE_ALL_DATAPOINTS:
            # --- ALL-DATA MODE: fit on ALL and write to original filenames (legacy behavior) ---
            y_all = Y[:, col_idx]

            # LASSO -> OLS on ALL
            res_all = lasso_then_ols(X, y_all, feature_names=feature_names)
            with open(os.path.join(out_dir, f"lasso_ols_{_safe_name(t)}.txt"), "w", encoding="utf-8") as fh:
                fh.write(f"# Target: {t} (ALL data)\n")
                fh.write(f"LASSO alpha: {res_all['lasso_alpha']}\n")
                fh.write("Selected features:\n")
                for fn in res_all["selected_features"]:
                    fh.write(f"  - {fn}\n")
                fh.write("\n--- OLS Summary (ALL) ---\n")
                fh.write(res_all["ols_summary"])

            # RF permutation importance on ALL (fit+permute on ALL as before)
            imp_all = random_forest_importance(X, y_all, feature_names=feature_names)
            imp_all.to_csv(os.path.join(out_dir, f"rf_permutation_importance_{_safe_name(t)}.csv"), index=False)

        else:
            # --- HOLDOUT MODE: fit on TRAIN; original filenames carry VALIDATION results; *_train.* carry training outputs ---
            y_tr = Y_train[:, col_idx]
            y_va = Y_val[:,   col_idx]

            # 1) LASSO -> OLS (fit on TRAIN)
            res_tr = lasso_then_ols(X_train, y_tr, feature_names=feature_names)

            # TRAIN file (separate)
            with open(os.path.join(out_dir, f"lasso_ols_{_safe_name(t)}_train.txt"), "w", encoding="utf-8") as fh:
                fh.write(f"# Target: {t} (TRAIN set)\n")
                fh.write(f"LASSO alpha: {res_tr['lasso_alpha']}\n")
                fh.write("Selected features:\n")
                for fn in res_tr["selected_features"]:
                    fh.write(f"  - {fn}\n")
                fh.write("\n--- OLS Summary (TRAIN) ---\n")
                fh.write(res_tr["ols_summary"])

            # VALIDATION file (original filename): use the TRAINED model’s selected features to build an OLS and report VAL metrics
            # Refit OLS on TRAIN selected features for prediction on VAL
            if res_tr["selected_features"]:
                sel = [feature_names.index(fn) for fn in res_tr["selected_features"]]
                Xtr_sel = X_train[:, sel]
                Xva_sel = X_val[:, sel]
                Xtr_ols = sm.add_constant(Xtr_sel, has_constant='add')
                ols_val_model = sm.OLS(y_tr, Xtr_ols, missing="drop").fit()
                y_hat_val = ols_val_model.predict(sm.add_constant(Xva_sel, has_constant='add'))
            else:
                # No features selected: predict mean of TRAIN
                y_hat_val = np.full_like(y_va, fill_value=np.mean(y_tr), dtype=float)

            with open(os.path.join(out_dir, f"lasso_ols_{_safe_name(t)}.txt"), "w", encoding="utf-8") as fh:
                fh.write(f"# Target: {t} (VALIDATION set)\n")
                fh.write("== Metrics on VALIDATION ==\n")
                fh.write(_metrics_report(y_va, y_hat_val))
                fh.write("\n(Trained via LASSO→OLS on TRAIN; selected features listed below for traceability.)\n")
                fh.write("Selected features (from TRAIN):\n")
                for fn in res_tr["selected_features"]:
                    fh.write(f"  - {fn}\n")

            # 2) Random Forest permutation importance
            # Fit RF on TRAIN
            rf_tr = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
            rf_tr.fit(X_train, y_tr)

            # TRAIN importances (separate file)
            from sklearn.inspection import permutation_importance as _perm
            perm_tr = _perm(rf_tr, X_train, y_tr, n_repeats=20, random_state=42, n_jobs=-1)
            imp_tr = pd.DataFrame({
                "predictor": feature_names,
                "perm_importance_mean": perm_tr.importances_mean,
                "perm_importance_std":  perm_tr.importances_std
            }).sort_values(by="perm_importance_mean", ascending=False)
            imp_tr.to_csv(os.path.join(out_dir, f"rf_permutation_importance_{_safe_name(t)}_train.csv"), index=False)

            # VALIDATION importances (original filename): compute permutation *on validation set*
            perm_va = _perm(rf_tr, X_val, y_va, n_repeats=20, random_state=42, n_jobs=-1)
            imp_va = pd.DataFrame({
                "predictor": feature_names,
                "perm_importance_mean": perm_va.importances_mean,
                "perm_importance_std":  perm_va.importances_std
            }).sort_values(by="perm_importance_mean", ascending=False)
            imp_va.to_csv(os.path.join(out_dir, f"rf_permutation_importance_{_safe_name(t)}.csv"), index=False)


    # Simple README
    with open(os.path.join(out_dir, "README_RESULTS.txt"), "w", encoding="utf-8") as fh:
        fh.write(
            "MFD FPCA-style Statistical Analysis Results\n"
            "===========================================\n\n"
            "- responses.csv includes all 7 MFD outputs per city.\n"
            "- No graph/topology features are used; all-NaN predictor columns are dropped.\n"
            "- Any remaining sporadic NaNs in predictors are median-imputed to keep modeling stable.\n"
        )

    # Status table
    status_df = pd.DataFrame(statuses, columns=["status","city","error"])
    status_df.to_csv(os.path.join(out_dir, "processing_status.csv"), index=False)
    print(f"Done. Results in: {out_dir}")
    return {
        "predictors_df": predictors_df,
        "responses_df": responses_df,
        "flow_df": flow_df,
        "scores_df": scores_df,
        "status": status_df
    }

# ------------------------
# Run (edit paths, then execute cell)
# ------------------------
OUT_DIR  = os.path.join(RESULTS, "StatResults")

results = run_pipeline(
    out_dir=OUT_DIR,
    n_pca=3,
    normalize_by_qc=True,
    force_recompute=False
)